# Merge Refinitiv & Trucost database
## Import libraries

In [38]:
import pandas as pd
import numpy as np

## Functions

In [39]:
# Function to rename a string named 'FY0' to '2022', 'FY-1' to '2021', 'FY-2' to '2020', etc.
def rename_fiscal_year(fiscal_year):
    current_year = 2022
    if fiscal_year == 'FY0':
        return str(current_year)
    elif fiscal_year.startswith('FY-'):
        year_offset = int(fiscal_year[3:])
        renamed_year = current_year - year_offset
        return str(renamed_year)
    else:
        return fiscal_year
    

## Refinitiv controls
### Import data

In [40]:
# Importing data from Refinitiv.
# Has 2 levels of column names and 6 index to decribe a company.
refinitiv_controls = pd.read_excel('data/Refinitiv_ Controls.xlsx', sheet_name="All controls_Agri", header=[0,1], index_col=[0,1,2,3,4,5])
# Rename the index because they disappeared.
refinitiv_controls.index.names=['Identifier (RIC)','Company Name','Ticker Symbol','Date Became Public','NAICS Industry Group Name','Business Description']

### Clean data

In [41]:
# Remove SD column
refinitiv_controls.drop('Earnings Per Share - Standard Deviation\n(USD)\nIn the last 18 Y',axis=1,inplace=True, level=0)
# Rename the column names based on fiscal years
refinitiv_controls.rename(columns=rename_fiscal_year, inplace=True, level=1)
# Remove year 2022
refinitiv_controls.drop('2022',axis=1,inplace=True, level=1)


### Reshape data

In [42]:
refinitiv_controls_vertical = refinitiv_controls.stack(level=1, dropna=False)

## Refinitiv returns
### Import data

It's necesserry to add year/month in the column header for returns per month
![Change in columns header](assets\returns_month.png)

In [47]:
refinitiv_returns = pd.read_excel('data/Refinitiv_ Returns.xlsx', sheet_name="Agriculture Comp, Full + Ticker", header=[0,1,2], index_col=[0,1,2,3,4,5])

### Clean data

In [48]:
# Remove return per year columns
refinitiv_returns.drop('Total Return by Year (01.01.2004-31.12.2022)',axis=1,inplace=True, level=0)
# Remove year 2022
refinitiv_returns.drop(2022,axis=1,inplace=True, level=1)

### Generate yearly returns statistics

In [54]:
# Gerenate column with available fiscal months per year
refinitiv_returns_count = refinitiv_returns.groupby(axis=1, level=1).count()
# Add header name
refinitiv_returns_count = pd.concat([refinitiv_returns_count], axis=1, keys=["Available fiscal months"])

In [50]:
# Generate mean for each year
refinitiv_returns_mean = refinitiv_returns.groupby(axis=1, level=1).mean()
# Add header name
refinitiv_returns_mean = pd.concat([refinitiv_returns_mean], axis=1, keys=["Mean"])


In [97]:
# Generate composed return for each year
refinitiv_returns_composed = (refinitiv_returns+1).groupby(axis=1, level=1).prod()-1
# Add header name
refinitiv_returns_composed = pd.concat([refinitiv_returns_composed], axis=1, keys=["Composed Return"])


In [98]:
refinitiv_returns_composed

Composed Return  \
                                                                                                                                                                               2004   
SIFB.BR  Sipef NV                           1987-01-05 Oilseed and Grain Farming                          Sipef NV is a Belgium-based agro industrial com... SIP          -0.040769   
CLEX.BR  Cumulex NV                         1987-01-06 Other Crop Farming                                 Cumulex NV, formerly Sucrerie et Raffinerie de ... CLEX          0.028571   
FMC.N    FMC Corp                           1948-09-10 Pesticide, Fertilizer, and Other Agricultural C... FMC Corporation is an agricultural sciences com... FMC           0.415177   
KUKZ.NR  Kakuzi PLC                         1995-04-24 Other Crop Farming                                 Kakuzi PLC is a Kenya-based agricultural compan... KUKZ          0.666667   
GENP.KL  Genting Plantations Bhd            1982-08-30 Oilseed and Grain Farming                          Genting Plantations Berhad is a Malaysia-based ... GENP          0.067492   
...                                                                                                                                                                             ...   
NIRM.NS  Nirman Agri Genetics Ltd           2023-03-28 Oilseed and Grain Farming                          Nirman Agri Genetics Ltd. is an India-based Agr... NIRMAN        0.000000   
PROP.BO  Prospect Commodities Ltd           2023-03-20 Fruit and Tree Nut Farming                         Prospect Commodities Limited is an India-based ... 543814        0.000000   
OHT.L    Ocean Harvest Technology Group PLC 2023-04-04 Aquaculture                                        Ocean Harvest Technology Group plc is a United ... OHT           0.000000   
ALFLO.PA Florentaise SA                     2023-04-12 Pesticide, Fertilizer, and Other Agricultural C... Florentaise SA is a France-based company that i... ALFLO         0.000000   
6952.TWO Dawushan Farm Technology Co Ltd    2023-06-21 Poultry and Egg Production                         Dawushan Farm Technology Co Ltd is a Taiwan-bas... 6952          0.000000   

                                                                                                                                                                               \
                                                                                                                                                                         2005   
SIFB.BR  Sipef NV                           1987-01-05 Oilseed and Grain Farming                          Sipef NV is a Belgium-based agro industrial com... SIP     0.367646   
CLEX.BR  Cumulex NV                         1987-01-06 Other Crop Farming                                 Cumulex NV, formerly Sucrerie et Raffinerie de ... CLEX   -0.083333   
FMC.N    FMC Corp                           1948-09-10 Pesticide, Fertilizer, and Other Agricultural C... FMC Corporation is an agricultural sciences com... FMC     0.100828   
KUKZ.NR  Kakuzi PLC                         1995-04-24 Other Crop Farming                                 Kakuzi PLC is a Kenya-based agricultural compan... KUKZ    0.206250   
GENP.KL  Genting Plantations Bhd            1982-08-30 Oilseed and Grain Farming                          Genting Plantations Berhad is a Malaysia-based ... GENP    0.218433   
...                                                                                                                                                                       ...   
NIRM.NS  Nirman Agri Genetics Ltd           2023-03-28 Oilseed and Grain Farming                          Nirman Agri Genetics Ltd. is an India-based Agr... NIRMAN  0.000000   
PROP.BO  Prospect Commodities Ltd           2023-03-20 Fruit and Tree Nut Farming                         Prospect Commodities Limited is an India-based ... 543814  0.000000   
OHT.L    Ocean

## Trucost
### Import data